In [8]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
# pd.reset_option()

In [9]:
df = pd.read_parquet(
    "s3://dyrmgraph/invalid/silver/gkg/pub_date=20261022", storage_options={"key": "dg_mi_poc_user", "secret": "dg_mi_poc_pw", "client_kwargs": {"endpoint_url": "http://localhost:9000"}}
)

In [ ]:
df.count()

In [11]:
regex_rules = {
    "GKGRecordID": r"^\d{14}-T?\d+$",
    "V2_1Date": r"^(?:\d{14}|0)$",
    "V2SourceCollectionIdentifier": r"^\d+$",
    "V2SourceCommonName": r"^[^\t]*$",
    "V2DocumentIdentifier": r"^[^\t]*$",
    "V1Counts": r"^(?:([^#;]*#){9}[^#;]*(?:;([^#;]*#){9}[^#;]*)*+;?)?$",
    "V2_1Counts": r"^(?:([^#;]*#){10}[^#;]*(?:;([^#;]*#){10}[^#;]*)*+;?)?$",
    "V1Themes": r"^(?:[^;]*(?:;[^;]*)*+)?$",
    "V2EnhancedThemes": r"^(?:[^,;]*,[^,;]*(?:;[^,;]*,[^,;]*)*+;?)?$",
    "V1Locations": r"^(?:([^#;]*#){6,7}[^#;]*(?:;([^#;]*#){6,7}[^#;]*)*+)?$",
    "V2EnhancedLocations": r"^(?:([^#;]*#){8}[^#;]*(?:;([^#;]*#){8}[^#;]*)*+)?$",
    "V1Persons": r"^(?:[^;]*(?:;[^;]*)*+)?$",
    "V2EnhancedPersons": r"^(?:[^,;]*,[^,;]*(?:;[^,;]*,[^,;]*)*+)?$",
    "V1Organizations": r"^(?:[^;]*(?:;[^;]*)*+)?$",
    "V2EnhancedOrganizations": r"^(?:[^,;]*,[^,;]*(?:;[^,;]*,[^,;]*)*+)?$",
    "V1_5Tone": r"^(?:([^,]*,){6}[^,]*)?$",
    "V2_1EnhancedDates": r"^(?:([^#;]*#){4}[^#;]*(?:;([^#;]*#){4}[^#;]*)*+)?$",
    "V2GCAM": r"^(?:[^,:]+:[^,]*(?:,[^,:]+:[^,]*)*+)?$",
    "V2_1SharingImage": r"^[^\t]*$",
    "V2_1RelatedImages": r"^(?:[^;]*(?:;[^;]*)*)?$",
    "V2_1SocialImageEmbeds": r"^(?:[^;]*(?:;[^;]*)*)?$",
    "V2_1SocialVideoEmbeds": r"^(?:[^;]*(?:;[^;]*)*)?$",
    "V2_1Quotations": r"^[^\t]*$",
    "V2_1AllNames": r"^(?:[^,;]*,[^,;]*(?:;[^,;]*,[^,;]*)*+)?$",
    "V2_1Amounts": r"^(?:([^,;]*,){2}[^,;]*(?:;([^,;]*,){2}[^,;]*)*+;?)?$",
    "V2_1TranslationInfo": r"^[^\t]*$",
    "V2ExtraSXML": r"^[^\t]*$",
}

In [14]:
invalid_results = []

for column, pattern in regex_rules.items():
    if column not in df.columns:
        print(f"Missing column: {column}")
        continue

    # Null values are trusted
    valid = df[column].astype("string[python]").str.fullmatch(pattern).fillna(True)

    invalid = df.loc[~valid, [column]].copy()

    if not invalid.empty:
        invalid_results.append(pd.DataFrame({"row_index": invalid.index, "column": column, "invalid_value": invalid[column].to_numpy()}))

invalid_df = pd.concat(invalid_results, ignore_index=True) if invalid_results else pd.DataFrame(columns=["row_index", "column", "invalid_value"])

print(f"Total invalid values: {len(invalid_df)}")
display(invalid_df)

Total invalid values: 1


row_index               column  \
0          0  V2EnhancedLocations   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [31]:
import re

res = re.fullmatch(regex_rules["V2EnhancedLocations"], invalid_df.loc[0]["invalid_value"])
print(res)

None


In [35]:
bad_value = invalid_df.loc[0]["invalid_value"]
nines = 0
eights = 0
nines_list = []
for block in bad_value.split(";"):
    if block.count("#") == 9:
        nines += 1
        nines_list.append(block)
    elif block.count("#") == 8:
        eights += 1
nines_list

['0#Georgia, , Georgia#GG#GG##42#.5#1#GG#148',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#223',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#625',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#685',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#981',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#1061',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#1300',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#1410',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#1446',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#2465',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#2659',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#3346',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#3629',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#3963',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#4107',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#4247',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#4441',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#5467',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#5808',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#5902',
 '0#Georgia, , Georgia#GG#GG##42#.5#1#GG#6292

In [9]:
summary = invalid_df.groupby("column").size().reset_index(name="invalid_count").sort_values("invalid_count", ascending=False)

display(summary)

,column,invalid_count
3,V2EnhancedThemes,1162
4,V2_1Amounts,941
6,V2_1EnhancedDates,506
0,V1Counts,149
5,V2_1Counts,149
2,V2EnhancedLocations,1
1,V1Locations,1
